# 13_log_transform.ipynb

**Experimento: Log-transform del target variable**

`total_reviews` tiene distribución muy sesgada (media=16, max=3759, mayoría <20).
Predecir `log(1 + reviews)` en vez de reviews crudos puede mejorar R² significativamente.

## Modelos testeados (re-entreno con log-target):
- **13a**: RS Embeddings only — vs Modelo 03 (R²=-0.027)
- **13b**: Hybrid (RS+TF-IDF+Numeric) — vs Modelo 08 (R²=-0.024)
- **13c**: Stage 2 Content (sin RS) — vs Modelo 12 (R²=0.076)

## Métricas reportadas:
- R²_log: en espacio logarítmico (métrica principal de comparación)
- RMSE_original: en escala original (back-transform con exp(pred)-1)
- R²_original: R² en escala original de las predicciones back-transformadas

In [1]:
import pandas as pd
import numpy as np
import json, ast, sys, os, warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

INTERACTIONS = "../Data/interactions.parquet"
ITEMMAP      = "../Data/item2idx.json"
STEAM_GAMES  = "../Data/steam_games.json"
RAWG_CSV     = "../Data/rawg_enriched.csv"
ITEM_EMB     = "../Data/item_embeddings_rs_clean.npy"
DEV_REP      = "../Data/developer_reputation.npy"
CUTOFF       = pd.to_datetime('2016-01-01')

with open(ITEMMAP, "r") as f:
    item2idx = {k: int(v) for k, v in json.load(f).items()}
N = len(item2idx)

item_emb = np.load(ITEM_EMB)   # (N, 64)
dev_rep  = np.load(DEV_REP)    # (N,)

df_inter  = pd.read_parquet(INTERACTIONS)
target_df = df_inter.groupby('item_idx').size().reset_index(name='total_reviews')
y_raw = target_df.set_index('item_idx').reindex(range(N), fill_value=0)['total_reviews'].values
y_log = np.log1p(y_raw.astype(float))   # log-transformed target

print(f"Target original: min={y_raw.min()} max={y_raw.max()} mean={y_raw.mean():.1f} std={y_raw.std():.1f}")
print(f"Target log:      min={y_log.min():.3f} max={y_log.max():.3f} mean={y_log.mean():.3f} std={y_log.std():.3f}")

Target original: min=1 max=3759 mean=16.1 std=107.8
Target log:      min=0.693 max=8.232 mean=1.568 std=1.140


In [2]:
def parse_list_col(x):
    if x is None or (isinstance(x, float) and pd.isna(x)): return []
    if isinstance(x, list): return [str(t).strip() for t in x if t]
    try:
        lst = eval(x)
        return [str(t).strip() for t in lst if t] if isinstance(lst, list) else []
    except:
        return []

def _parse_price(p):
    try: return float(p)
    except: return 0.0

# Steam metadata + TF-IDF
games = []
with open(STEAM_GAMES, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try: games.append(ast.literal_eval(line))
        except: pass

df_games = pd.json_normalize(games).rename(columns={'id': 'item_id'})
df_games['item_idx'] = df_games['item_id'].map(item2idx)
df_games = df_games.dropna(subset=['item_idx'])
df_games['item_idx'] = df_games['item_idx'].astype(int)
df_games['release_date_parsed'] = pd.to_datetime(df_games['release_date'], errors='coerce')
df_games['tag_text']    = df_games['tags'].apply(parse_list_col).apply(' '.join)
df_games['genre_text']  = df_games['genres'].apply(parse_list_col).apply(' '.join)
df_games['content_text']= df_games['tag_text'] + ' ' + df_games['genre_text']
df_games['price_num']   = df_games['price'].apply(_parse_price)
df_games['ea_flag']     = df_games['early_access'].apply(lambda x: 1 if x else 0)

# temporal masks (all-items indexed)
item_dates = df_games.set_index('item_idx')['release_date_parsed'].reindex(range(N))
train_mask = np.array([pd.notna(d) and d < CUTOFF  for d in item_dates])
test_mask  = np.array([pd.notna(d) and d >= CUTOFF for d in item_dates])

# TF-IDF
games_w_content = df_games[df_games['content_text'].str.strip() != ''].copy()
tfidf = TfidfVectorizer(max_features=100, min_df=2, max_df=0.5, ngram_range=(1,1))
# Fit TF-IDF only on pre-2016 games to avoid leakage from test set
_pre2016_content = games_w_content[
    games_w_content['release_date_parsed'] < CUTOFF
]['content_text']
tfidf.fit(_pre2016_content)
tfidf_matrix = tfidf.transform(games_w_content['content_text'])
item_to_tfidf = {int(r['item_idx']): i for i, (_, r) in enumerate(games_w_content.iterrows())}

# RAWG
df_rawg = pd.read_csv(RAWG_CSV).drop_duplicates(subset=['item_idx'], keep='last').set_index('item_idx')
for col in ['genres','platforms','developers','publishers','tags']:
    if col in df_rawg.columns:
        df_rawg[col] = df_rawg[col].apply(parse_list_col)

ESRB_ORDER = {'Everyone':1,'Everyone 10+':2,'Teen':3,'Mature':4,'Adults Only':5}

def rawg_vec(idx):
    if idx not in df_rawg.index: return [0.]*12
    r = df_rawg.loc[idx]
    return [
        1.0 if r.get('rawg_id') is not None else 0.0,
        float(r['rawg_rating'])       if pd.notna(r.get('rawg_rating'))       else -1.0,
        0.0 if pd.notna(r.get('rawg_rating'))       else 1.0,
        float(r['metacritic'])         if pd.notna(r.get('metacritic'))        else -1.0,
        0.0 if pd.notna(r.get('metacritic'))        else 1.0,
        np.log1p(float(r['playtime_avg_h']))     if pd.notna(r.get('playtime_avg_h'))     else 0.0,
        np.log1p(float(r['rawg_ratings_count'])) if pd.notna(r.get('rawg_ratings_count')) else 0.0,
        float(len(parse_list_col(r.get('platforms',[])))),
        float(len(parse_list_col(r.get('genres',[])))),
        float(len(parse_list_col(r.get('developers',[])))),
        float(len(parse_list_col(r.get('publishers',[])))),
        float(ESRB_ORDER.get(r.get('esrb_rating',''), 0)),
    ]

rawg_arr = np.array([rawg_vec(i) for i in range(N)], dtype=np.float32)
print(f"TF-IDF: {tfidf_matrix.shape} | RAWG: {rawg_arr.shape}")
print(f"Train: {train_mask.sum()} | Test: {test_mask.sum()}")


TF-IDF: (3194, 100) | RAWG: (3682, 12)
Train: 2621 | Test: 486


In [3]:
TSCV_WINDOWS = [
    ('2013-07-01','2014-01-01'),('2014-01-01','2014-07-01'),
    ('2014-07-01','2015-01-01'),('2015-01-01','2015-07-01'),
    ('2015-07-01','2016-01-01'),
]

def run_log_experiment(X_feat, y_log_arr, y_raw_arr, tr_mask, te_mask,
                       item_dates_arr, valid_mask,
                       model_id, model_name, features_desc, emb_type,
                       rawg_impute_cols=None, rawg_offset=None):
    """Train on log-target. Report R²_log and back-transformed R²/RMSE."""

    X_tr, y_tr = X_feat[tr_mask], y_log_arr[tr_mask]
    X_te, y_te_log = X_feat[te_mask], y_log_arr[te_mask]
    y_te_raw = y_raw_arr[te_mask]

    # Impute RAWG -1 with train median
    if rawg_impute_cols and rawg_offset is not None:
        for ci in rawg_impute_cols:
            col = rawg_offset + ci
            tv = X_tr[:, col]; vv = tv[tv != -1.]
            if len(vv): X_feat[X_feat[:, col] == -1., col] = float(np.median(vv))
        X_tr, X_te = X_feat[tr_mask], X_feat[te_mask]

    print(f"\n{'='*70}")
    print(f"[{model_id}] {model_name} — LOG TARGET")
    print(f"Features: {features_desc} | Shape: {X_feat.shape}")
    print(f"Train: {len(X_tr)} | Test: {len(X_te)}")
    print(f"{'='*70}")

    # Optuna
    # Sort by release date so Optuna val set is always the most recent 20%
    _train_dates = item_dates_arr[tr_mask]
    _sort_order = np.argsort(_train_dates)
    X_tr = X_tr[_sort_order]
    y_tr = y_tr[_sort_order]

    sp = max(10, int(len(X_tr)*0.8))
    X_opt, X_val = X_tr[:sp], X_tr[sp:]
    y_opt, y_val = y_tr[:sp], y_tr[sp:]

    def objective(trial):
        p = dict(
            n_estimators=trial.suggest_int('n_estimators',100,800),
            learning_rate=trial.suggest_float('learning_rate',1e-3,0.3,log=True),
            max_depth=trial.suggest_int('max_depth',3,8),
            min_child_weight=trial.suggest_int('min_child_weight',1,10),
            subsample=trial.suggest_float('subsample',0.5,1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree',0.5,1.0),
            gamma=trial.suggest_float('gamma',0.0,2.0),
            random_state=42, tree_method='hist', verbosity=0,
        )
        m = XGBRegressor(**p)
        m.fit(X_opt, y_opt)
        return float(mean_squared_error(y_val, m.predict(X_val))**0.5)

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=50, show_progress_bar=True)
    best_params = {**study.best_params, 'random_state':42, 'tree_method':'hist', 'verbosity':0}
    print(f"Best val RMSE_log: {study.best_value:.4f}")

    # Train + predict
    model = XGBRegressor(**best_params)
    model.fit(X_tr, y_tr)
    pred_log = model.predict(X_te)
    pred_raw = np.expm1(pred_log).clip(0)  # back-transform

    r2_log      = r2_score(y_te_log, pred_log)
    rmse_log    = mean_squared_error(y_te_log, pred_log)**0.5
    r2_raw      = r2_score(y_te_raw, pred_raw)
    rmse_raw    = mean_squared_error(y_te_raw, pred_raw)**0.5
    mae_raw     = mean_absolute_error(y_te_raw, pred_raw)
    mape_raw    = mean_absolute_percentage_error(y_te_raw, pred_raw)

    print(f"R²_log (log-space):       {r2_log:.6f}")
    print(f"R²_original (back-transf):{r2_raw:.6f}")
    print(f"RMSE_log:                 {rmse_log:.4f}")
    print(f"RMSE_original:            {rmse_raw:.2f}")

    # TSCV on log target
    ts_results = []
    for tc_str, sc_str in TSCV_WINDOWS:
        tc, sc = pd.Timestamp(tc_str), pd.Timestamp(sc_str)
        t_msk = np.array([pd.notna(d) and d < tc for d in item_dates_arr])
        e_msk = np.array([pd.notna(d) and d >= tc and d < sc for d in item_dates_arr])
        if e_msk.sum() < 5: continue
        Xf = X_feat.copy()
        if rawg_impute_cols and rawg_offset is not None:
            for ci in rawg_impute_cols:
                col = rawg_offset+ci
                tv = Xf[t_msk,col]; vv = tv[tv!=-1.]
                if len(vv): Xf[Xf[:,col]==-1., col] = float(np.median(vv))
        m_ts = XGBRegressor(**best_params)
        m_ts.fit(Xf[t_msk], y_log_arr[t_msk])
        p_ts = m_ts.predict(Xf[e_msk])
        ts_results.append({'R2': r2_score(y_log_arr[e_msk], p_ts),
                           'RMSE': mean_squared_error(y_log_arr[e_msk], p_ts)**0.5})
    ts_df = pd.DataFrame(ts_results)
    r2_tscv_mean = ts_df['R2'].mean(); r2_tscv_std = ts_df['R2'].std()
    rmse_tscv_mean = ts_df['RMSE'].mean(); rmse_tscv_std = ts_df['RMSE'].std()
    print(f"TSCV R²_log: {r2_tscv_mean:.4f} ± {r2_tscv_std:.4f}")

    # KFold
    X_kf = X_feat[valid_mask]; y_kf = y_log_arr[valid_mask]
    kf_res = []
    for ti, ei in KFold(n_splits=5, shuffle=True, random_state=42).split(X_kf):
        m_kf = XGBRegressor(**best_params)
        m_kf.fit(X_kf[ti], y_kf[ti])
        p_kf = m_kf.predict(X_kf[ei])
        kf_res.append({'R2': r2_score(y_kf[ei], p_kf),
                       'RMSE': mean_squared_error(y_kf[ei], p_kf)**0.5})
    kf_df = pd.DataFrame(kf_res)
    r2_kfold_mean = kf_df['R2'].mean(); r2_kfold_std = kf_df['R2'].std()
    rmse_kfold_mean = kf_df['RMSE'].mean()
    print(f"KFold R²_log: {r2_kfold_mean:.4f} ± {r2_kfold_std:.4f}")

    # Save
    sys.path.insert(0, os.path.abspath('.'))
    from results_tracker import save_result
    save_result(
        model_id=model_id,
        model_name=model_name,
        features=features_desc,
        embeddings=emb_type,
        metrics={
            'r2_temporal':   r2_raw,   # back-transformed (comparable with linear models)
        'r2_temporal_log': r2_log, # in log-space (for reference)       'rmse_temporal': rmse_log,
            'mae_temporal':  mae_raw,      'mape_temporal': mape_raw,
            'r2_tscv':       r2_tscv_mean, 'r2_tscv_std':  r2_tscv_std,
            'rmse_tscv':     rmse_tscv_mean,'rmse_tscv_std':rmse_tscv_std,
            'r2_kfold':      r2_kfold_mean, 'r2_kfold_std': r2_kfold_std,
            'rmse_kfold':    rmse_kfold_mean,
        },
        notes=f'LOG TARGET. R2_log={r2_log:.4f} R2_orig={r2_raw:.4f} RMSE_orig={rmse_raw:.2f}',
    )
    print(f"Saved [{model_id}]")
    return model, r2_log, r2_raw, r2_tscv_mean

print('Runner ready')


Runner ready


In [4]:
# ── 13a: RS Embeddings only + log target ───────────────────────────────────
valid_mask_all = train_mask | test_mask
item_dates_all = np.array(item_dates.values)

X_rs = item_emb.copy()   # (N, 64)
model_13a, r2log_13a, r2raw_13a, tscv_13a = run_log_experiment(
    X_rs, y_log, y_raw, train_mask, test_mask,
    item_dates_all, valid_mask_all,
    '13a', 'RS Only (log target)', 'RS clean (64d) [log-target]', 'clean'
)


[13a] RS Only (log target) — LOG TARGET
Features: RS clean (64d) [log-target] | Shape: (3682, 64)
Train: 2621 | Test: 486


  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.326243:   0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.326243:   2%|▏         | 1/50 [00:00<00:10,  4.61it/s]

Best trial: 1. Best value: 0.283319:   2%|▏         | 1/50 [00:00<00:10,  4.61it/s]

...

Best trial: 49. Best value: 0.262301: 100%|██████████| 50/50 [00:24<00:00,  2.08it/s]

Best val RMSE_log: 0.2623


R²_log (log-space):       -0.292201
R²_original (back-transf):-0.026268
RMSE_log:                 1.1104
RMSE_original:            57.64


TSCV R²_log: 0.9369 ± 0.0093


KFold R²_log: 0.8161 ± 0.0426
Saved [13a]


In [5]:
# ── 13b: Hybrid (RS+TF-IDF+Numeric) + log target ──────────────────────────
# Rebuild hybrid features same as model 08
hybrid_feats, hybrid_idxs = [], []
for idx in range(N):
    if idx not in item_to_tfidf: continue
    g = games_w_content[games_w_content['item_idx']==idx]
    if g.empty: continue
    g = g.iloc[0]
    tfidf_vec = tfidf_matrix[item_to_tfidf[idx]].toarray().flatten()
    vec = np.concatenate([item_emb[idx], tfidf_vec, [g['price_num'], g['ea_flag']]])
    hybrid_feats.append(vec)
    hybrid_idxs.append(idx)

X_hyb = np.array(hybrid_feats, dtype=np.float32)
y_log_hyb = np.log1p(target_df.set_index('item_idx').reindex(hybrid_idxs, fill_value=0)['total_reviews'].values.astype(float))
y_raw_hyb = np.expm1(y_log_hyb).astype(int)

data_df_hyb = pd.DataFrame({'item_idx':hybrid_idxs}).merge(
    df_games[['item_idx','release_date_parsed']], on='item_idx', how='left'
)
hyb_dates   = data_df_hyb['release_date_parsed'].values
hyb_train   = data_df_hyb['release_date_parsed'].notna() & (data_df_hyb['release_date_parsed'] < CUTOFF)
hyb_test    = data_df_hyb['release_date_parsed'].notna() & (data_df_hyb['release_date_parsed'] >= CUTOFF)
hyb_valid   = data_df_hyb['release_date_parsed'].notna()
hyb_train, hyb_test, hyb_valid = hyb_train.values, hyb_test.values, hyb_valid.values

model_13b, r2log_13b, r2raw_13b, tscv_13b = run_log_experiment(
    X_hyb, y_log_hyb, y_raw_hyb, hyb_train, hyb_test,
    hyb_dates, hyb_valid,
    '13b', 'Hybrid RS+TF-IDF (log target)', 'RS (64d)+TF-IDF (100d)+Numeric (2d) [log-target]', 'clean'
)


[13b] Hybrid RS+TF-IDF (log target) — LOG TARGET
Features: RS (64d)+TF-IDF (100d)+Numeric (2d) [log-target] | Shape: (3194, 166)
Train: 2620 | Test: 486


  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.308958:   0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.308958:   2%|▏         | 1/50 [00:00<00:16,  2.96it/s]

Best trial: 1. Best value: 0.284167:   2%|▏         | 1/50 [00:01<00:16,  2.96it/s]

...

Best trial: 49. Best value: 0.258904: 100%|██████████| 50/50 [00:41<00:00,  1.20it/s]

Best val RMSE_log: 0.2589


R²_log (log-space):       -0.253018
R²_original (back-transf):-0.024792
RMSE_log:                 1.0934
RMSE_original:            57.56


TSCV R²_log: 0.9403 ± 0.0083


KFold R²_log: 0.8557 ± 0.0323
Saved [13b]


In [6]:
# ── 13c: Stage 2 Content (sin RS) + log target ────────────────────────────
stage2_rows, s2_idxs = [], []
for idx in range(N):
    if idx not in item_to_tfidf: continue
    g = games_w_content[games_w_content['item_idx']==idx]
    if g.empty: continue
    g = g.iloc[0]
    tfidf_vec = tfidf_matrix[item_to_tfidf[idx]].toarray().flatten()
    vec = np.concatenate([tfidf_vec, [g['price_num'], g['ea_flag']], rawg_arr[idx], [dev_rep[idx]]])
    stage2_rows.append(vec)
    s2_idxs.append(idx)

X_s2 = np.array(stage2_rows, dtype=np.float32)
y_log_s2 = np.log1p(target_df.set_index('item_idx').reindex(s2_idxs, fill_value=0)['total_reviews'].values.astype(float))
y_raw_s2 = np.expm1(y_log_s2).astype(int)

data_df_s2 = pd.DataFrame({'item_idx':s2_idxs}).merge(
    df_games[['item_idx','release_date_parsed']], on='item_idx', how='left'
)
s2_dates  = data_df_s2['release_date_parsed'].values
s2_train  = data_df_s2['release_date_parsed'].notna() & (data_df_s2['release_date_parsed'] < CUTOFF)
s2_test   = data_df_s2['release_date_parsed'].notna() & (data_df_s2['release_date_parsed'] >= CUTOFF)
s2_valid  = data_df_s2['release_date_parsed'].notna()
s2_train, s2_test, s2_valid = s2_train.values, s2_test.values, s2_valid.values

RAWG_OFFSET_S2 = 102  # after tfidf(100)+steam(2)

model_13c, r2log_13c, r2raw_13c, tscv_13c = run_log_experiment(
    X_s2, y_log_s2, y_raw_s2, s2_train, s2_test,
    s2_dates, s2_valid,
    '13c', 'Stage2 Content (log target)', 'TF-IDF (100d)+Steam (2d)+RAWG (12d)+dev_rep (1d) [log-target]', 'none',
    rawg_impute_cols=[1,3], rawg_offset=RAWG_OFFSET_S2
)


[13c] Stage2 Content (log target) — LOG TARGET
Features: TF-IDF (100d)+Steam (2d)+RAWG (12d)+dev_rep (1d) [log-target] | Shape: (3194, 115)
Train: 2620 | Test: 486


  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.538057:   0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.538057:   2%|▏         | 1/50 [00:00<00:18,  2.72it/s]

Best trial: 1. Best value: 0.482389:   2%|▏         | 1/50 [00:01<00:18,  2.72it/s]

...

Best trial: 23. Best value: 0.465636: 100%|██████████| 50/50 [00:34<00:00,  1.44it/s]

Best val RMSE_log: 0.4656


R²_log (log-space):       0.038832
R²_original (back-transf):0.049504
RMSE_log:                 0.9577
RMSE_original:            55.43


TSCV R²_log: 0.7545 ± 0.0383


KFold R²_log: 0.7190 ± 0.0191
Saved [13c]


In [7]:
sys.path.insert(0, os.path.abspath('.'))
from results_tracker import print_leaderboard
print_leaderboard()

print("\n" + "="*70)
print("ABLACION: LOG-TRANSFORM vs TARGET LINEAL")
print("="*70)
print(f"{'Modelo':<30s}  {'R²_log':>8s}  {'R²_orig':>8s}  {'TSCV R²_log':>12s}")
print("-"*65)
print(f"{'03 RS Only (lineal)':<30s}  {'N/A':>8s}  {-0.0270:>8.4f}  {0.7699:>12.4f}")
print(f"{'13a RS Only (log)':<30s}  {r2log_13a:>8.4f}  {r2raw_13a:>8.4f}  {tscv_13a:>12.4f}")
print()
print(f"{'08 Hybrid (lineal)':<30s}  {'N/A':>8s}  {-0.0240:>8.4f}  {0.8410:>12.4f}")
print(f"{'13b Hybrid (log)':<30s}  {r2log_13b:>8.4f}  {r2raw_13b:>8.4f}  {tscv_13b:>12.4f}")
print()
print(f"{'12 Stage2 Content (lineal)':<30s}  {'N/A':>8s}  {0.0760:>8.4f}  {-5.028:>12.4f}")
print(f"{'13c Stage2 Content (log)':<30s}  {r2log_13c:>8.4f}  {r2raw_13c:>8.4f}  {tscv_13c:>12.4f}")
print("="*70)
print("\nNota: R²_log = R² calculado en espacio log(1+y)")
print("      R²_orig = R² de predicciones back-transformadas (exp(pred)-1) vs y_raw")

 ID  Modelo                         R2 test     RMSE     MAE   MAPE%   R2 TSCV  R2 KFold
[12]  Two-Stage: Content Model        0.0737    54.76   13.97     4.7    0.4686    0.2580
[04]  Metadata Only                   0.0572    55.24   18.84     8.4   -0.2144    0.0710
[13c]  Stage2 Content (log target)     0.0495      N/A   10.38     1.3    0.7545    0.7190
[06]  Review Text Emb                -0.0107    57.20   18.76     8.3   -0.1720    0.0072
[05]  RS + Metadata                  -0.0161    57.35    9.16     0.6    0.8888    0.0110
[11b]  Hybrid + Dev Reputation        -0.0163    57.36    9.40     0.8    0.8580    0.3073
[10]  RS + Reviews + RAWG            -0.0163    57.36    9.14    46.6    0.8634    0.4330
[03]  RS Embeddings Only             -0.0184    57.42    9.64     1.1    0.8861    0.4510
[09]  RS + Review Text               -0.0213    57.50    9.27    49.1    0.8733    0.3359
[08]  Hybrid Collab-Content          -0.0240    57.57    9.37     0.4    0.8562    0.2945
[13b]  Hy